# 📱 Read the vibe: RNN vs LSTM vs GRU on real text

Same text, three brains. You'll train a **vanilla RNN**, an **LSTM** and a **GRU** on real human writing and see which one actually *gets* it.

Pick your vibe check in the settings cell 👇
- **`goemotions`**: Reddit comments → 7 emotions (anger, disgust, fear, joy, sadness, surprise, neutral)
- **`tweet_emoji`**: tweets → which of 20 emojis fits 🔥😂❤
- **`imdb`**: long movie reviews → positive / negative

**What you'll do**
1. Load + explore the data 🔍
2. Turn text into numbers (done for you, but read it!)
3. ✍️ TODO: build `SequenceClassifier` (embedding → RNN/LSTM/GRU → head)
4. ✍️ TODO: write the training loop + accuracy
5. 🏁 Race all three with the same budget, plot it, talk about **vanishing gradients**
6. 🎤 Bonus: type your own sentence
7. 🚀 Submit your score to DL Lab

**How to run:** `Runtime → Change runtime type → T4 GPU`, fill the settings, then `Runtime → Run all`. Every `# TODO` has a ✅ check cell right after it. If a check fails, read the hint, fix, re-run.

In [ ]:
#@title ⚙️ Settings: paste your platform details, pick a dataset
PLATFORM_URL = ""   #@param {type:"string"}
SUBMIT_TOKEN = ""   #@param {type:"string"}
DATASET = "goemotions"  #@param ["goemotions", "tweet_emoji", "imdb"]
QUICK = False  #@param {type:"boolean"}

In [ ]:
#@title 🧰 Setup: install `datasets`, imports, pick a device
!pip install -q datasets

import copy, random, re, time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset

def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    # Apple's MPS is skipped on purpose: recurrent layers run slower there than on the CPU
    return torch.device("cpu")

device = pick_device()
print("🖥️ device:", device, "| torch", torch.__version__, "| dataset:", DATASET, "| QUICK:", QUICK)
if device.type != "cuda":
    print("⚠️ No GPU here. On Colab: Runtime → Change runtime type → T4 GPU (then re-run).")
    if not QUICK:
        print("   Or tick QUICK in the settings cell for a tiny CPU-friendly run.")

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 1. 🔍 Load the data

| `DATASET` | Source (Hugging Face) | Task | Max tokens |
|---|---|---|---|
| `goemotions` | `google-research-datasets/go_emotions` (Apache-2.0) | Reddit comment → 1 of 7 Ekman emotions | 40 |
| `tweet_emoji` | `cardiffnlp/tweet_eval`, config `emoji` | tweet → 1 of 20 emojis | 40 |
| `imdb` | `stanfordnlp/imdb` | movie review → pos / neg | 200 |

- **goemotions** originally has 28 fine-grained labels and comments can have several. We keep comments with **exactly one** label and map the 28 labels onto the 7 **Ekman** groups with the official mapping (it's the `EKMAN` dict below).
- **tweet_emoji**: the dataset's license is listed as *unknown*, so treat it as **classroom use only** 🏫.
- **imdb**: reviews are *long* (hundreds of words). That's exactly where the vanilla RNN's **vanishing gradient** hurts: the signal from early words has to survive ~200 multiplications to reach the end. Watch for it! 👀 (imdb has no validation split, so we carve 10% off train.)

We use the official train / validation / test splits. With `QUICK=True` we only take a small random subset of each.

In [ ]:
# Official GoEmotions → Ekman mapping (28 labels → 6 Ekman emotions + neutral)
EKMAN = {
    "anger":    ["anger", "annoyance", "disapproval"],
    "disgust":  ["disgust"],
    "fear":     ["fear", "nervousness"],
    "joy":      ["joy", "amusement", "approval", "excitement", "gratitude", "love", "optimism",
                 "relief", "pride", "admiration", "desire", "caring"],
    "sadness":  ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral":  ["neutral"],
}

SUBSET = dict(train=3000, val=500, test=1000) if QUICK else dict(train=None, val=None, test=None)

def take(ds, n):
    """Random subset of n examples (shuffle first: imdb is sorted by label!)."""
    if n is not None and len(ds) > n:
        ds = ds.shuffle(seed=SEED).select(range(n))
    return ds

def clean(text):
    return text.replace("<br />", " ")   # imdb has HTML line breaks

if DATASET == "goemotions":
    raw = load_dataset("google-research-datasets/go_emotions", "simplified")
    fine_names = raw["train"].features["labels"].feature.names
    LABEL_NAMES = list(EKMAN)
    fine_to_ekman = {fine_names.index(f): LABEL_NAMES.index(g) for g, fs in EKMAN.items() for f in fs}
    assert len(fine_to_ekman) == 28
    splits = {"train": raw["train"], "val": raw["validation"], "test": raw["test"]}
    splits = {k: v.filter(lambda ex: len(ex["labels"]) == 1) for k, v in splits.items()}   # single-label only
    get_label = lambda ex: fine_to_ekman[ex["labels"][0]]
    MAX_LEN, TASK = 40, "emotion"
elif DATASET == "tweet_emoji":
    raw = load_dataset("cardiffnlp/tweet_eval", "emoji")
    LABEL_NAMES = raw["train"].features["label"].names
    splits = {"train": raw["train"], "val": raw["validation"], "test": raw["test"]}
    get_label = lambda ex: ex["label"]
    MAX_LEN, TASK = 40, "emoji"
elif DATASET == "imdb":
    raw = load_dataset("stanfordnlp/imdb")
    LABEL_NAMES = ["negative", "positive"]
    carved = raw["train"].train_test_split(test_size=0.1, seed=SEED, stratify_by_column="label")
    splits = {"train": carved["train"], "val": carved["test"], "test": raw["test"]}
    get_label = lambda ex: ex["label"]
    MAX_LEN, TASK = 200, "sentiment"
else:
    raise ValueError(f"Unknown DATASET {DATASET!r}")

data = {}
for name, ds in splits.items():
    ds = take(ds, SUBSET[name])
    data[name] = ([clean(t) for t in ds["text"]], [get_label(ex) for ex in ds])

train_texts, train_labels = data["train"]
val_texts, val_labels = data["val"]
test_texts, test_labels = data["test"]
NUM_CLASSES = len(LABEL_NAMES)
print(f"train {len(train_texts):,} | val {len(val_texts):,} | test {len(test_texts):,} | {NUM_CLASSES} classes")

In [ ]:
#@title 👀 Peek at the data + class balance
rng = random.Random(1)
for i in rng.sample(range(len(train_texts)), 5):
    t = train_texts[i]
    print(f"[{LABEL_NAMES[train_labels[i]]}]  {t[:160]}{'…' if len(t) > 160 else ''}")

counts = Counter(train_labels)
print("\nClass balance (train):")
for c in range(NUM_CLASSES):
    print(f"  {c:2d} {LABEL_NAMES[c]:>10s}  {counts[c]:6,}  ({100 * counts[c] / len(train_labels):5.1f}%)")
majority = counts.most_common(1)[0][0]
BASELINE = sum(y == majority for y in test_labels) / len(test_labels)
print(f"\n🎯 'Always guess {LABEL_NAMES[majority]}' gets {100 * BASELINE:.1f}% test accuracy. Your models must beat this!")

# emoji glyphs often don't render in matplotlib fonts → use indices for tweet_emoji
tick_names = [str(c) for c in range(NUM_CLASSES)] if DATASET == "tweet_emoji" else LABEL_NAMES
plt.figure(figsize=(8, 3))
plt.bar(tick_names, [counts[c] for c in range(NUM_CLASSES)], color="#7c5cff")
plt.title(f"{DATASET}: train label counts"); plt.ylabel("examples")
plt.xticks(rotation=45 if NUM_CLASSES > 2 else 0); plt.tight_layout(); plt.show()
if DATASET == "tweet_emoji":
    print("label index → emoji:", "  ".join(f"{i}:{e}" for i, e in enumerate(LABEL_NAMES)))

## 2. 🔢 Text → numbers (provided)

Neural nets eat numbers, not words. The recipe:
1. **Tokenise**: lowercase, then grab words with a regex (`[a-z0-9']+`) and treat each punctuation mark / emoji as its own token.
2. **Vocab**: keep the most common words from the *train* split. Index `0 = <pad>`, `1 = <unk>` (any word we never saw).
3. **Encode + truncate** each text to at most `MAX_LEN` ids.
4. **Pad** each batch to the length of its longest sequence with `<pad>`, and remember the real `lengths`. 🧠 Remember those lengths, you'll need them in the model!

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9']+|[^\sa-z0-9']")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

VOCAB_SIZE = 20_000 if DATASET == "imdb" else 15_000
PAD, UNK = 0, 1
word_counts = Counter(tok for t in train_texts for tok in tokenize(t))
itos = ["<pad>", "<unk>"] + [w for w, c in word_counts.most_common(VOCAB_SIZE - 2) if c >= 2]
stoi = {w: i for i, w in enumerate(itos)}

def encode(text):
    ids = [stoi.get(tok, UNK) for tok in tokenize(text)][:MAX_LEN]
    return ids or [UNK]          # never return an empty sequence

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.ids = [encode(t) for t in texts]
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return self.ids[i], self.labels[i]

def collate(batch):
    """List of (ids, label) → x (B, T) padded with PAD, lengths (B,), y (B,)."""
    lengths = torch.tensor([len(ids) for ids, _ in batch])
    x = torch.full((len(batch), int(lengths.max())), PAD, dtype=torch.long)
    for row, (ids, _) in enumerate(batch):
        x[row, :len(ids)] = torch.tensor(ids)
    y = torch.tensor([label for _, label in batch])
    return x, lengths, y

BATCH_SIZE = 64
train_ds, val_ds, test_ds = TextDataset(train_texts, train_labels), TextDataset(val_texts, val_labels), TextDataset(test_texts, test_labels)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=256, collate_fn=collate)
test_loader = DataLoader(test_ds, batch_size=256, collate_fn=collate)

print(f"vocab size: {len(itos):,}")
print("example:", tokenize(train_texts[0])[:15])
print("ids:    ", encode(train_texts[0])[:15])
x, lengths, y = next(iter(train_loader))
print("one batch → x", tuple(x.shape), "| lengths", lengths[:8].tolist(), "… | y", y[:8].tolist(), "…")

raw_lens = [len(tokenize(t)) for t in train_texts]
plt.figure(figsize=(7, 2.8))
plt.hist(raw_lens, bins=50, color="#22c55e"); plt.axvline(MAX_LEN, color="k", ls="--", label=f"MAX_LEN={MAX_LEN}")
plt.title(f"tokens per text ({100 * np.mean(np.array(raw_lens) > MAX_LEN):.0f}% get truncated)"); plt.legend(); plt.show()

## 3. ✍️ TODO: the model

On the platform you built these by hand in NumPy: `rnn_step`, `lstm_cell`, `gru_cell`. PyTorch ships them, already looped over time:

| `cell` | PyTorch | One step (what you coded on the platform) |
|---|---|---|
| `"rnn"` | `nn.RNN` | `rnn_step`: h' = tanh(Wx·x + Wh·h + b) |
| `"lstm"` | `nn.LSTM` | `lstm_cell`: c' = f⊙c + i⊙g, h' = o⊙tanh(c') |
| `"gru"` | `nn.GRU` | `gru_cell`: h' = (1−z)⊙n + z⊙h |

With `batch_first=True`, `out, _ = self.rnn(emb)` gives `out` of shape **(B, T, H)**: the hidden state after **every** step.

🚨 **The padding trap:** `out[:, -1]` is the state after the last *column*, and for short texts that column is `<pad>`. The RNN has been chewing on padding! We want the state at the **last real token**, position `lengths - 1` in each row.
Hint: advanced indexing `out[rows, cols]` with `rows = torch.arange(B)` and `cols = lengths - 1` picks one time step per row → shape (B, H).

(The fancy alternative is `nn.utils.rnn.pack_padded_sequence`; indexing is simpler and gives the same answer for a one-direction RNN.)

Fill in the three blanks:
1. pick the RNN class for `cell` (hint: the `RNN_TYPES` dict)
2. the head: a linear layer from `hidden` to `num_classes`
3. `h_last`: the hidden state at the last real token

In [ ]:
RNN_TYPES = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}

class SequenceClassifier(nn.Module):
    """tokens → embedding → RNN / LSTM / GRU → hidden state at the last real token → linear head → logits"""

    def __init__(self, vocab_size, num_classes, cell="lstm", emb_dim=128, hidden=128):
        super().__init__()
        self.cell = cell
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        rnn_cls = ...  # TODO: pick nn.RNN / nn.LSTM / nn.GRU for this `cell`
        self.rnn = rnn_cls(emb_dim, hidden, batch_first=True)
        self.head = ...  # TODO: linear layer hidden → num_classes

    def forward(self, x, lengths):
        # x: (B, T) token ids, lengths: (B,) number of real (non-pad) tokens per row
        emb = self.embedding(x)                    # (B, T, E)
        out, _ = self.rnn(emb)                     # (B, T, H) hidden state after every step
        lengths = lengths.to(out.device)
        rows = torch.arange(out.size(0), device=out.device)
        h_last = ...  # TODO: (B, H) hidden state at the LAST REAL token of each row
        return self.head(h_last)                   # (B, num_classes) logits

In [ ]:
#@title ✅ Check your model
torch.manual_seed(0)
for cell, cls in [("rnn", nn.RNN), ("lstm", nn.LSTM), ("gru", nn.GRU)]:
    m = SequenceClassifier(50, 5, cell=cell, emb_dim=8, hidden=16)
    assert isinstance(m.rnn, cls), f"cell='{cell}' should give self.rnn = {cls.__name__}, got {type(m.rnn).__name__}. Hint: RNN_TYPES[cell]"
    assert isinstance(m.head, nn.Linear) and (m.head.in_features, m.head.out_features) == (16, 5), \
        "self.head should be nn.Linear(hidden, num_classes)"
    x = torch.randint(2, 50, (3, 7)); lengths = torch.tensor([7, 4, 2])
    x[1, 4:] = PAD; x[2, 2:] = PAD
    logits = m(x, lengths)
    assert logits.shape == (3, 5), f"logits should be (batch, num_classes) = (3, 5), got {tuple(logits.shape)}"
    x_more_pad = torch.cat([x, torch.full((3, 5), PAD)], dim=1)
    assert torch.allclose(m(x_more_pad, lengths), logits, atol=1e-5), \
        "Extra padding changed the output 😬. Hint: take out[rows, lengths - 1], not out[:, -1]."
    assert torch.allclose(m(x[2:3, :2], lengths[2:3]), logits[2:3], atol=1e-5), \
        "A short text should give the same logits alone as inside a padded batch."
print("✅ nice: all three cells build, give (B, num_classes) logits, and ignore padding")

## 4. ✍️ TODO: train + evaluate

The classic five-step training dance, once per batch:
1. **forward**: `logits = model(x, lengths)`
2. **loss**: cross-entropy between `logits` and the true labels `y`
3. **`zero_grad`**: wipe old gradients (PyTorch *adds* them up otherwise)
4. **`backward`**: backprop through time 🔙
5. **`clip_grad_norm_`** then **`step`**: clipping caps the gradient norm so RNNs don't *explode* (it can't fix *vanishing* though!)

Then `evaluate` = fraction of examples where the arg-max of the logits matches the label.

In [ ]:
def train_one_epoch(model, loader, optimizer, clip=1.0):
    model.train()
    total_loss, n = 0.0, 0
    for x, lengths, y in loader:
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)
        logits = ...  # TODO: forward pass
        loss = ...  # TODO: cross-entropy loss
        ...  # TODO: reset gradients
        ...  # TODO: backprop through time
        ...  # TODO: clip the gradient norm to `clip`
        ...  # TODO: update the weights
        total_loss += loss.item() * len(y)
        n += len(y)
    return total_loss / n

@torch.no_grad()
def evaluate(model, loader):
    """Accuracy in [0, 1]."""
    model.eval()
    correct, total = 0, 0
    for x, lengths, y in loader:
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)
        logits = model(x, lengths)
        preds = ...  # TODO: predicted class per example
        correct += ...  # TODO: how many predictions are right (a Python int)
        total += len(y)
    return correct / total

In [ ]:
#@title ✅ Check your training loop
torch.manual_seed(0)
toy_x = torch.randint(2, 20, (64, 6))
toy_y = (toy_x == 5).any(dim=1).long()           # toy task: "is token 5 in the sentence?"
toy_loader = DataLoader(list(zip(toy_x, torch.full((64,), 6), toy_y)), batch_size=16)

class AlwaysZero(nn.Module):
    def forward(self, x, lengths):
        return torch.tensor([[1.0, 0.0]], device=x.device).repeat(len(x), 1)
acc0 = evaluate(AlwaysZero(), toy_loader)
assert isinstance(acc0, float), f"evaluate should return a float, got {type(acc0).__name__}"
assert abs(acc0 - (toy_y == 0).float().mean().item()) < 1e-6, "evaluate: a model that always says class 0 should score the share of class-0 labels"

calls = Counter()
_real_clip = torch.nn.utils.clip_grad_norm_
def _counting_clip(*a, **k):
    calls["clip"] += 1
    return _real_clip(*a, **k)
torch.nn.utils.clip_grad_norm_ = _counting_clip
try:
    toy = SequenceClassifier(20, 2, "gru", emb_dim=16, hidden=32).to(device)
    opt = torch.optim.Adam(toy.parameters(), lr=1e-2)
    for name in ["zero_grad", "step"]:
        real = getattr(opt, name)
        setattr(opt, name, (lambda real, name: lambda *a, **k: (calls.update([name]), real(*a, **k))[1])(real, name))
    losses = [train_one_epoch(toy, toy_loader, opt) for _ in range(25)]
finally:
    torch.nn.utils.clip_grad_norm_ = _real_clip
assert isinstance(losses[0], float), "train_one_epoch should return the average loss as a float"
assert calls["zero_grad"] == calls["step"] == calls["clip"] == 25 * 4, \
    f"expected zero_grad, clip_grad_norm_ and step once per batch, got {dict(calls)}"
assert losses[-1] < 0.5 * losses[0], f"loss should drop a lot on the toy task ({losses[0]:.3f} → {losses[-1]:.3f}). Order: zero_grad → backward → clip → step"
assert evaluate(toy, toy_loader) > 0.9, "the toy model should get >90% after training"
print(f"✅ nice: toy loss {losses[0]:.3f} → {losses[-1]:.3f}, toy accuracy {evaluate(toy, toy_loader):.0%}")

## 5. 🏁 The race: RNN vs LSTM vs GRU

Same data, same embedding size, same hidden size, same optimizer, same number of epochs. Only the recurrent cell changes. After each epoch we check validation accuracy and keep the best checkpoint, then score it **once** on the test set.

⏱️ On a T4 this takes roughly 10–15 minutes for all three (less for the short-text datasets). `QUICK=True` does 1 tiny epoch.

In [ ]:
#@title 🏋️ Train all three
EPOCHS = 1 if QUICK else {"goemotions": 10, "tweet_emoji": 6, "imdb": 10}[DATASET]
EMB, HIDDEN, LR = (64, 64, 2e-3) if QUICK else (128, 256, 2e-3)

results = {}
for cell in ["rnn", "lstm", "gru"]:
    torch.manual_seed(SEED)
    model = SequenceClassifier(len(itos), NUM_CLASSES, cell=cell, emb_dim=EMB, hidden=HIDDEN).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n🧠 {cell.upper()}  ({n_params:,} params)")
    hist = {"train_loss": [], "val_acc": []}
    best_val, best_state, t0 = -1.0, None, time.time()
    for epoch in range(1, EPOCHS + 1):
        loss = train_one_epoch(model, train_loader, optimizer)
        val_acc = evaluate(model, val_loader)
        hist["train_loss"].append(loss); hist["val_acc"].append(val_acc)
        if val_acc > best_val:
            best_val, best_state = val_acc, copy.deepcopy(model.state_dict())
        print(f"  epoch {epoch:2d}  train loss {loss:.3f}  val acc {100 * val_acc:5.1f}%  ({time.time() - t0:.0f}s)")
    model.load_state_dict(best_state)
    test_acc = evaluate(model, test_loader)
    results[cell] = dict(model=model, hist=hist, best_val=best_val, test_acc=test_acc,
                         params=n_params, seconds=time.time() - t0)
    print(f"  ➜ best val {100 * best_val:.1f}%  |  test {100 * test_acc:.1f}%")

In [ ]:
#@title 📊 Plot the race (saves result.png)
COLORS = {"rnn": "#ef4444", "lstm": "#3b82f6", "gru": "#22c55e"}
cells = list(results)

plt.figure(figsize=(7, 4))
for c in cells:
    v = results[c]["hist"]["val_acc"]
    plt.plot(range(1, len(v) + 1), [100 * a for a in v], "o-", color=COLORS[c], label=c.upper())
plt.axhline(100 * BASELINE, color="gray", ls=":", label="majority guess")
plt.xlabel("epoch"); plt.ylabel("validation accuracy (%)"); plt.title(f"{DATASET}: validation accuracy")
plt.legend(); plt.grid(alpha=0.3); plt.show()

best_cell = max(cells, key=lambda c: results[c]["test_acc"])
best_test_acc = 100 * results[best_cell]["test_acc"]

fig = plt.figure(figsize=(6, 4))
accs = [100 * results[c]["test_acc"] for c in cells]
bars = plt.bar([c.upper() for c in cells], accs, color=[COLORS[c] for c in cells])
for b, a in zip(bars, accs):
    plt.text(b.get_x() + b.get_width() / 2, a + 0.5, f"{a:.1f}%", ha="center", fontweight="bold")
plt.axhline(100 * BASELINE, color="gray", ls=":", label=f"majority guess ({100 * BASELINE:.1f}%)")
plt.ylabel("test accuracy (%)"); plt.ylim(0, max(accs + [100 * BASELINE]) * 1.15)
plt.title(f"{DATASET} ({TASK}): RNN vs LSTM vs GRU" + ("  [QUICK]" if QUICK else ""))
plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig("result.png", dpi=100, bbox_inches="tight")
plt.show()

print(f"{'cell':6s} {'params':>10s} {'time':>7s} {'best val':>9s} {'test':>7s}")
for c in cells:
    r = results[c]
    print(f"{c:6s} {r['params']:>10,} {r['seconds']:>6.0f}s {100 * r['best_val']:>8.1f}% {100 * r['test_acc']:>6.1f}%")
print(f"\n🏆 best: {best_cell.upper()} with {best_test_acc:.1f}% test accuracy (saved result.png)")

### 🔬 How far back can each model "hear"?

On the platform (`bptt` step) you watched the gradient shrink as it flows back through `tanh` steps. Let's measure it on the **trained** models: take some long validation texts, backprop the loss, and look at the gradient size reaching the word embedding at each position. Position 0 = the last token; further right = further in the past. If a curve hits the floor (1e-30), the gradient has vanished so hard that float32 rounds it to exactly zero 🫠.

In [ ]:
#@title 🔬 Gradient reaching each position (vanishing-gradient check)
def grad_through_time(model, x, lengths, y):
    store = {}
    def hook(module, inputs, output):
        output.retain_grad(); store["emb"] = output
    handle = model.embedding.register_forward_hook(hook)
    model.train()   # cuDNN only backprops RNNs in train mode (no dropout here, so same outputs)
    model.zero_grad()
    F.cross_entropy(model(x, lengths), y).backward()
    handle.remove(); model.zero_grad()
    g = store["emb"].grad.norm(dim=2).mean(dim=0)       # (T,) average gradient size per position
    return g.flip(0).cpu().numpy()                       # index 0 = last token

order = sorted(range(len(val_ds)), key=lambda i: -len(val_ds.ids[i]))[:64]
L = min(len(val_ds.ids[i]) for i in order)               # crop everything to a common length
probe = collate([(val_ds.ids[i][:L], val_ds.labels[i]) for i in order])
x, lengths, y = (t.to(device) for t in probe)

FLOOR = 1e-30   # float32 underflows to exactly 0 somewhere below this → "fully vanished"
steps_back = [k for k in (5, 10, 25, 50, 100, 150) if k < L]
print("gradient k tokens back, relative to the last token:")
print("       " + "".join(f"{k:>9d}" for k in steps_back))
plt.figure(figsize=(7, 4))
for c in cells:
    g = grad_through_time(results[c]["model"], x, lengths, y)
    rel = g.astype(np.float64) / g[0]
    plt.semilogy(np.maximum(rel, FLOOR), color=COLORS[c], label=c.upper())
    print(f"{c:5s}  " + "".join(f"{rel[k]:9.0e}" if rel[k] > FLOOR else f"{'~0':>9s}" for k in steps_back))
plt.xlabel("tokens before the end"); plt.ylabel("‖∂loss/∂embedding‖ (relative, floored at 1e-30)")
plt.title(f"How much does each position still matter? (L={L})"); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 💬 Discussion: why does the vanilla RNN lag?

- The vanilla RNN does `h' = tanh(Wx·x + Wh·h + b)` at every step (your `rnn_step`). Backprop through time multiplies one Jacobian `diag(1 − h²)·Wh` **per step**. When those factors are "smaller than 1", the gradient from the loss shrinks **exponentially** on its way back → words far from the end barely get a learning signal. That's the **vanishing gradient** (see the red curve above dropping fastest).
- The **LSTM** writes memory with an **additive** update `c' = f⊙c + i⊙g`. If the forget gate `f ≈ 1`, the gradient flows back through `c` almost untouched: a gradient highway 🛣️.
- The **GRU** does the same trick with fewer parameters: `h' = (1−z)⊙n + z⊙h` lets it copy the old state forward when `z ≈ 1`.
- Gradient **clipping** only fixes *exploding* gradients. It can't make a vanished gradient come back.
- **imdb** is where this really bites: ~200 tokens, and the sentiment clue ("…but honestly, a masterpiece") can be anywhere. For short comments and tweets (≤ 40 tokens) the RNN often keeps up much better. Try both and compare!

🤔 **Think about it:** Which cell won on your dataset, and by how much? Was the gap bigger on `imdb`? Does the LSTM's extra parameters (4 gates vs GRU's 3) buy you anything here?

## 🎤 Bonus: try your own sentence

Type anything (a text to your friend, a tweet draft, a hot take on a movie) and see what the best model thinks. Compare with the other two cells too.

In [ ]:
#@title 🎤 What's the vibe?
MY_TEXT = "i can't believe they cancelled my favourite show, this is the worst"  #@param {type:"string"}

@torch.no_grad()
def predict(text, model, k=3):
    model.eval()
    ids = encode(text)
    x = torch.tensor([ids], device=device)
    probs = F.softmax(model(x, torch.tensor([len(ids)])), dim=1)[0].cpu()
    top = probs.topk(min(k, NUM_CLASSES))
    return [(LABEL_NAMES[i], p.item()) for p, i in zip(top.values, top.indices)]

ids = encode(MY_TEXT)
print(f"📝 {MY_TEXT!r}  ({len(ids)} tokens, {sum(i == UNK for i in ids)} unknown)\n")
for c in [best_cell] + [c for c in cells if c != best_cell]:
    preds = predict(MY_TEXT, results[c]["model"])
    star = " 🏆" if c == best_cell else ""
    print(f"{c.upper():5s}{star}: " + "   ".join(f"{name} {100 * p:.0f}%" for name, p in preds))

## 🚀 Submit

Paste `PLATFORM_URL` and `SUBMIT_TOKEN` (from the challenge page on DL Lab) into the settings cell at the top, run it, then run the cell below. It sends your best test accuracy and `result.png`.

In [ ]:
#@title 🚀 Submit your result to DL Lab
import base64, json, urllib.request, urllib.error
CHALLENGE_ID = "sequences"   # "sequences" | "attention" | "gans"

def submit(metric_name=None, metric=None, image_path=None, notes=""):
    if not PLATFORM_URL or not SUBMIT_TOKEN:
        print("⚠️ Paste PLATFORM_URL and SUBMIT_TOKEN into the settings cell at the top (copy them from the platform's challenge page).")
        return
    body = {"dataset": DATASET, "metric_name": metric_name, "metric": None if metric is None else float(metric), "notes": notes}
    if image_path:
        with open(image_path, "rb") as f:
            body["image_b64"] = base64.b64encode(f.read()).decode()
    req = urllib.request.Request(PLATFORM_URL.rstrip("/") + f"/api/challenges/{CHALLENGE_ID}/submit",
                                 data=json.dumps(body).encode(), method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": "Bearer " + SUBMIT_TOKEN.strip()})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            print("🎉", json.loads(r.read())["message"])
    except urllib.error.HTTPError as e:
        print("❌ Submit failed:", e.code, e.read().decode()[:300])
    except Exception as e:
        print("❌ Could not reach the platform:", e)

submit(metric_name="test_accuracy", metric=best_test_acc, image_path="result.png", notes=f"best={best_cell}")